<div style="font-family:-apple-system,Segoe UI,sans-serif;background:linear-gradient(108deg,#004F49 0 52%,#EB6658 52%);color:#fff;border-radius:16px;padding:26px 30px;">
<div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:.9;">International Summer School on Generative AI · Rome 2026</div>
<h1 style="margin:8px 0 4px;font-size:34px;">From Graph to Crew — The World Cup Lab</h1>
<div style="font-size:16px;opacity:.95;">A hands-on tour of agentic AI frameworks · Mohammad Dehghani · AI2Lab</div>
</div>

We build the **same task twice**: a World Cup match-analysis system that produces a scouting report — a *Scout* gathers data, a *Stats* and a *Tactics* analyst interpret it, and a *Writer* composes the report.

First with **LangGraph** (a *graph* you draw), then with **CrewAI** (a *crew* you assemble) — in that order, from **graph to crew**.

> **The goal is not the football.** It is to *feel* the difference between the two orchestration styles by building the identical task in both.

## How to use this notebook

1. Run the cells **top to bottom**. **After Step 1 (install), restart the runtime** — *Runtime ▸ Restart session* — then keep going. Installing CrewAI upgrades libraries Colab has already loaded, and skipping the restart is the usual cause of `ModuleNotFoundError: No module named 'crewai'`.
2. Each participant uses **their own API key**, stored as a **Colab secret** (the key icon in the left sidebar). Never paste a key into a shared notebook or commit it to GitHub.
3. The lab ships with **mock match data** so it runs even without web access. A stretch goal swaps in a real search tool.

**Step 2 auto-detects your provider** from the keys you've added — OpenAI, Gemini, or Anthropic. No key found? It will prompt you securely.

In [ ]:
# =========================================================
# Step 1 · Install the frameworks
# =========================================================
# IMPORTANT — after this cell finishes, RESTART the runtime
# (Runtime ▸ Restart session), then run the rest top-to-bottom.
# CrewAI upgrades libraries Colab already loaded; the restart prevents
# "ModuleNotFoundError: No module named 'crewai'".
%pip install crewai langgraph langchain-anthropic langchain-openai langchain-google-genai

In [ ]:
# =========================================================
# Step 2 · Auto-detect your provider from the keys you have
# =========================================================
import os, getpass

# Leave as None to auto-detect, or force one: "openai" | "google" | "anthropic".
FORCE_PROVIDER = None

MODELS = {"anthropic": "claude-haiku-4-5-20251001",
          "openai":    "gpt-4o-mini",
          "google":    "gemini-2.5-flash"}
KEY_ENV = {"anthropic": "ANTHROPIC_API_KEY",
           "openai":    "OPENAI_API_KEY",
           "google":    "GOOGLE_API_KEY"}
# CrewAI routes through LiteLLM, whose Google prefix is "gemini/" (not "google/").
LITELLM_PREFIX = {"anthropic": "anthropic", "openai": "openai", "google": "gemini"}

# When several keys are present, prefer in this order.
PREFERENCE = ["openai", "google", "anthropic"]

def _load_secret(name):
    """Return a secret from Colab Secrets or the environment, else None."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

# Discover which providers you actually have a key for.
available = {p: _load_secret(KEY_ENV[p]) for p in PREFERENCE}
available = {p: v for p, v in available.items() if v}

# Pick: manual override wins, else first available by preference, else prompt.
if FORCE_PROVIDER:
    PROVIDER = FORCE_PROVIDER
elif available:
    PROVIDER = next(p for p in PREFERENCE if p in available)
else:
    PROVIDER = "openai"

# Export the chosen provider's key (prompt securely if it wasn't found).
key_val = available.get(PROVIDER) or getpass.getpass(f"Enter your {KEY_ENV[PROVIDER]}: ")
os.environ[KEY_ENV[PROVIDER]] = key_val
# LiteLLM's Gemini provider reads GEMINI_API_KEY; mirror it from GOOGLE_API_KEY.
if PROVIDER == "google":
    os.environ["GEMINI_API_KEY"] = key_val

MODEL = MODELS[PROVIDER]
print(f"Keys found: {sorted(available) or 'none (prompted)'}")
print(f"Provider: {PROVIDER}  ·  Model: {MODEL}")

In [ ]:
# =========================================================
# Step 3 · A pretty-print helper for clean output in Colab
# =========================================================
from IPython.display import HTML, display

# Reports come back as Markdown (**bold**, lists, headings). Render that to HTML
# so the card shows clean formatting instead of literal * and # characters.
try:
    import markdown as _md
    def _md_to_html(text):
        return _md.markdown(text, extensions=["sane_lists", "tables"])
except Exception:
    import re
    NL = chr(10)
    def _md_to_html(text):
        text = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", text)
        text = re.sub(r"\*(.+?)\*", r"<i>\1</i>", text)
        text = re.sub(r"(?m)^#{1,6}\s*(.+)$", r"<h4 style='margin:10px 0 4px;'>\1</h4>", text)
        text = re.sub(r"(?m)^\s*[-*]\s+(.+)$", r"&bull; \1<br>", text)
        return text.replace(NL + NL, "<br><br>").replace(NL, "<br>")

def pretty_report(title, body, accent="#004F49"):
    """Render a titled card. `body` may be Markdown; it is converted to clean HTML."""
    display(HTML(f"""
    <div style="font-family:-apple-system,Segoe UI,sans-serif;border:1px solid #e3e0d8;
                border-left:6px solid {accent};border-radius:12px;padding:18px 22px;margin:10px 0;
                background:#fbfaf7;box-shadow:0 2px 10px rgba(20,30,40,.06);">
      <div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:{accent};
                  font-weight:700;margin-bottom:8px;">{title}</div>
      <div style="font-size:15px;line-height:1.6;color:#1a1a1a;">{_md_to_html(body)}</div>
    </div>"""))

pretty_report("Pretty print ready",
              "Reports render as clean cards — **bold**, lists, and headings come through tidy.")


In [ ]:
# =========================================================
# Step 4 · Data source: a tiny mock so the lab runs offline
# (Swap in a real web-search tool later — see the stretch goal.)
# =========================================================
MATCH_DATA = {
    ("Italy", "Brazil"): {
        "form":    "Italy: W W D L W   ·   Brazil: W W W D W",
        "stats":   "Italy 1.6 xG/game, compact mid-block.  Brazil 2.1 xG/game, high press.",
        "tactics": "Italy 3-5-2, patient build-up.  Brazil 4-2-3-1, full-back overlaps.",
    }
}

def get_match_data(team_a, team_b):
    """Return raw match facts for two teams (mock)."""
    return MATCH_DATA.get((team_a, team_b), {
        "form": "(mock) recent form unavailable",
        "stats": "(mock) stats unavailable",
        "tactics": "(mock) tactical notes unavailable"})

TEAM_A, TEAM_B = "Italy", "Brazil"
print(get_match_data(TEAM_A, TEAM_B))

## Build 1 · LangGraph — *a state machine you draw*

We start with the **graph**. You declare a shared **State**, write **nodes** that update it, and connect them with **edges** — including a **conditional edge** that can loop back to the scout when data is missing. Then we **render the graph** and watch it run node by node.

*If you can draw the flow, you can build the graph.*

In [ ]:
# =========================================================
# Build 1 · LangGraph — state, nodes, edges, a conditional loop
# =========================================================
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# One chat model, chosen from the provider you set in Step 2
if PROVIDER == "anthropic":
    from langchain_anthropic import ChatAnthropic
    chat = ChatAnthropic(model=MODEL, temperature=0)
elif PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    chat = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    chat = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

def ask(prompt):                      # tiny LLM helper
    return chat.invoke(prompt).content

# 1) State = a typed object every node reads and updates
class State(TypedDict):
    raw: str
    stats: str
    tactics: str
    report: str
    loops: int

# 2) Nodes = functions state -> partial state update (each prints so you can see it fire)
def scout_node(s):
    n = s.get("loops", 0) + 1
    print(f"  [scout]   pass #{n} — gathering match data")
    return {"raw": str(get_match_data(TEAM_A, TEAM_B)), "loops": n}

def stats_node(s):
    print("  [stats]   reading the numbers")
    return {"stats": ask(f"Give 3 concise statistical insights from: {s['raw']}")}

def tactics_node(s):
    print("  [tactics] reading shape and style")
    return {"tactics": ask(f"Give 3 concise tactical insights from: {s['raw']}")}

def write_node(s):
    print("  [write]   composing the report")
    return {"report": ask("Write a one-page football scouting report with a predicted edge.\n"
                          f"Stats: {s['stats']}\nTactics: {s['tactics']}")}

# 3) Conditional edge = loop back to scout if data is missing (capped, so it always ends)
def need_more(s):
    return "scout" if (s["loops"] < 2 and "unavailable" in s["raw"]) else "tactics"

# 4) Wire the graph: nodes, edges, the conditional loop, then compile
g = StateGraph(State)
g.add_node("scout", scout_node)
g.add_node("stats", stats_node)
g.add_node("tactics", tactics_node)
g.add_node("write", write_node)
g.add_edge(START, "scout")
g.add_edge("scout", "stats")
g.add_conditional_edges("stats", need_more, {"scout": "scout", "tactics": "tactics"})
g.add_edge("tactics", "write")
g.add_edge("write", END)
app = g.compile()
print("Graph compiled:  START -> scout -> stats -> (need more? -> scout) -> tactics -> write -> END")

In [ ]:
# =========================================================
# See the graph you drew — then watch it run, node by node
# =========================================================
from IPython.display import Image, display

# Render the compiled graph as a diagram (falls back to text if rendering is offline)
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Diagram service unavailable — here is the graph as Mermaid text:\n")
    print(app.get_graph().draw_mermaid())

# Run it. The node prints above will trace the path the state took.
print("\nRunning the graph:")
graph_out = app.invoke({"loops": 0})
pretty_report(f"LangGraph scouting report · {TEAM_A} vs {TEAM_B}", graph_out["report"], accent="#004F49")

### Your turn — tweak the graph

Change **one** thing below and re-run the cell. Ideas:
- **New matchup:** try an *unknown* pair like `"Spain", "Japan"`. There's no mock data for it, so `need_more` routes back to `scout` — watch the pass counter climb to the cap.
- **Loop cap:** in the Build 1 cell, change `s["loops"] < 2` to `< 3`, re-run that cell, then this one.
- **Prompt:** in `write_node`, ask for a *predicted scoreline*, re-run Build 1, then this one.

In [ ]:
# === YOUR TURN (graph) — change a value, then run this cell ===
TEAM_A, TEAM_B = "Italy", "Brazil"     # <-- try "Spain", "Japan" (no data -> the loop fires)

out = app.invoke({"loops": 0})
pretty_report(f"Your LangGraph run · {TEAM_A} vs {TEAM_B}", out["report"], accent="#004F49")

## Build 1 · Advanced — parallel analysts + a reflection loop

A more realistic graph that uses what LangGraph is *for*:
- **Parallelism:** `stats` and `tactics` run **at the same time** (fan-out from `scout`, fan-in to `write`).
- **Reflection:** a **Critic** node reviews the draft and can route it **back to the writer** to improve, until it is approved (capped, so it always ends).

In [ ]:
# =========================================================
# Build 1 · Advanced — parallel analysts + a critic that reflects
# =========================================================
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AdvState(TypedDict):
    raw: str
    stats: str
    tactics: str
    report: str
    verdict: str
    revisions: int

def a_scout(s):
    print("  [scout]   gathering data")
    return {"raw": str(get_match_data(TEAM_A, TEAM_B)), "revisions": 0}

def a_stats(s):
    print("  [stats]   (parallel) reading the numbers")
    return {"stats": ask(f"Give 3 concise statistical insights from: {s['raw']}")}

def a_tactics(s):
    print("  [tactics] (parallel) reading shape and style")
    return {"tactics": ask(f"Give 3 concise tactical insights from: {s['raw']}")}

def a_write(s):
    n = s.get("revisions", 0)
    fix = f"The editor asked you to fix: {s['verdict']}" if s.get("verdict") else ""
    print(f"  [write]   draft #{n + 1}")
    report = ask(f"""Write a one-page football scouting report with a predicted edge.
Use clean Markdown and no emojis.
Stats: {s['stats']}
Tactics: {s['tactics']}
{fix}""")
    return {"report": report, "revisions": n + 1}

def a_critic(s):
    print("  [critic]  reviewing the draft")
    verdict = ask(f"""You are a tough editor. If this scouting report is specific and decision-useful, reply with exactly APPROVED. Otherwise reply with ONE concrete fix.

{s['report']}""")
    return {"verdict": verdict}

def needs_revision(s):
    approved = "APPROVED" in s.get("verdict", "").upper()
    return "write" if (not approved and s["revisions"] < 2) else "done"

ag = StateGraph(AdvState)
ag.add_node("scout", a_scout)
ag.add_node("stats", a_stats)
ag.add_node("tactics", a_tactics)
ag.add_node("write", a_write)
ag.add_node("critic", a_critic)
ag.add_edge(START, "scout")
ag.add_edge("scout", "stats")      # fan-out: both analysts start after scout
ag.add_edge("scout", "tactics")
ag.add_edge("stats", "write")      # fan-in: write runs once, after BOTH analysts
ag.add_edge("tactics", "write")
ag.add_edge("write", "critic")
ag.add_conditional_edges("critic", needs_revision, {"write": "write", "done": END})
adv = ag.compile()
print("Advanced graph compiled: scout -> (stats || tactics) -> write <-> critic -> END")


In [ ]:
# See the advanced graph, then run it: stats & tactics fire together,
# then the critic loops the draft back to the writer until it is approved.
from IPython.display import Image, display
try:
    display(Image(adv.get_graph().draw_mermaid_png()))
except Exception:
    print(adv.get_graph().draw_mermaid())

print("Running the advanced graph:")
adv_out = adv.invoke({})
pretty_report(f"Advanced LangGraph report · {TEAM_A} vs {TEAM_B}", adv_out["report"], accent="#004F49")


## Build 2 · CrewAI — *a team you assemble*

The same task, now as a **crew**. You write four **job descriptions** (agents), give each a **task**, and let a **sequential process** run them in order. There is no graph to wire — you describe the team, and CrewAI coordinates them.

*If you can describe the team, you can build the crew.*

In [ ]:
# =========================================================
# Build 2 · CrewAI — agents, tasks, crew
# =========================================================
from crewai import Agent, Task, Crew, Process, LLM

# CrewAI talks to any provider through one model string, e.g. "anthropic/claude-..."
llm = LLM(model=f"{LITELLM_PREFIX[PROVIDER]}/{MODEL}")

data = get_match_data(TEAM_A, TEAM_B)

# 1) Agents = role + goal + backstory
scout   = Agent(role="Scout",           goal="Gather raw match data on both teams",
                backstory="A meticulous football scout.",      llm=llm, verbose=True)
stats   = Agent(role="Stats Analyst",   goal="Interpret the numbers",
                backstory="A data-driven analyst.",            llm=llm, verbose=True)
tactics = Agent(role="Tactics Analyst", goal="Read shape and style",
                backstory="A seasoned tactical expert.",       llm=llm, verbose=True)
writer  = Agent(role="Report Writer",   goal="Assemble a clear scouting report",
                backstory="A concise sports writer.",          llm=llm, verbose=True)

# 2) Tasks = description + expected_output + the agent that owns it
t_scout = Task(description=f"Collect data for {TEAM_A} vs {TEAM_B}. Raw facts:\n{data}",
               expected_output="A bullet list of raw facts.",            agent=scout)
t_stats = Task(description="From the scout's facts, find the statistical edge for each side.",
               expected_output="3 to 4 statistical insights.",          agent=stats)
t_tact  = Task(description="From the scout's facts, analyze tactical matchups and key duels.",
               expected_output="3 to 4 tactical insights.",             agent=tactics)
t_write = Task(description="Write a one-page scouting report with a predicted edge.",
               expected_output="A short markdown scouting report.",     agent=writer)

# 3) Crew = agents + tasks + a process, then kick it off
crew = Crew(agents=[scout, stats, tactics, writer],
            tasks=[t_scout, t_stats, t_tact, t_write],
            process=Process.sequential, verbose=True)

# Colab already runs an asyncio event loop, so CrewAI refuses a synchronous
# crew.kickoff() here. Use the async kickoff and await it (works in Colab cells).
crew_result = await crew.kickoff_async()
pretty_report(f"CrewAI scouting report · {TEAM_A} vs {TEAM_B}", str(crew_result), accent="#D9543F")

### Your turn — tweak the crew

Edit the **Build 2 · CrewAI** cell above and re-run it. Ideas:
- **Re-cast an agent:** change the `writer`'s `goal` or `backstory` (a tabloid headline writer vs. a sober tactical analyst) and feel the tone shift.
- **Change the ask:** in `t_write`, request a *predicted scoreline and a one-line headline*.
- **Sharpen a role:** give `stats` the goal "find the ONE decisive stat" and watch the report tighten.
- **New matchup:** set `TEAM_A, TEAM_B` and re-run from Build 2 (this drives both builds).

## Build 2 · Advanced — a custom tool + a hierarchical manager

A more realistic crew that uses what CrewAI is *for*:
- **Tools:** the Scout gets a real `@tool` it can call for a head-to-head record.
- **Hierarchical process:** a **manager** LLM plans the work and **delegates** to the agents instead of a fixed sequence — you describe the goal, the manager runs the team.

In [ ]:
# =========================================================
# Build 2 · Advanced — a custom tool + a hierarchical manager
# =========================================================
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

@tool("Head-to-head record")
def head_to_head(matchup: str) -> str:
    """Return the historical head-to-head record for a 'TeamA vs TeamB' matchup."""
    return ("Last 5 meetings: 2 wins each, 1 draw, avg 2.6 goals/game; "
            "the home side scored first in 4 of 5.")

scout_a   = Agent(role="Scout", goal="Gather match data and the head-to-head record",
                  backstory="A meticulous scout who checks the record books.",
                  tools=[head_to_head], llm=llm, verbose=True)
analyst_a = Agent(role="Analyst", goal="Turn the data into the single decisive insight",
                  backstory="A sharp analyst who cuts to what matters.", llm=llm, verbose=True)
writer_a  = Agent(role="Report Writer", goal="Write a crisp scouting verdict",
                  backstory="A concise sports writer.", llm=llm, verbose=True)

brief = Task(
    description=f"Produce a scouting verdict for {TEAM_A} vs {TEAM_B}: gather the data and "
                "head-to-head, find the decisive edge, and write a short report. "
                "Use clean Markdown and no emojis.",
    expected_output="A short Markdown scouting report ending in a predicted edge.")

# Hierarchical: the manager LLM plans and delegates (no fixed order, no agent per task)
crew_adv = Crew(agents=[scout_a, analyst_a, writer_a], tasks=[brief],
                process=Process.hierarchical,
                manager_llm=LLM(model=f"{LITELLM_PREFIX[PROVIDER]}/{MODEL}"),
                verbose=True)

crew_adv_result = await crew_adv.kickoff_async()
pretty_report(f"Advanced CrewAI report · {TEAM_A} vs {TEAM_B}", str(crew_adv_result), accent="#D9543F")


## Same task, two architectures — what did you feel?

| | **LangGraph** (graph) | **CrewAI** (crew) |
|---|---|---|
| You wrote | nodes + edges + state | role + goal + task |
| The loop | **explicit** conditional edge | hidden in the process |
| You can see | the graph you drew | the agents' dialogue |
| Best when | control, branching, loops | quick role decomposition |

**Pair, think, share:** which would you reach for in your own research, and why?

## Another layer — take the lab further

Pick one and extend the notebook:

**1. Give the Scout a real web-search tool** (replace the mock `get_match_data`):

```python
# %pip install duckduckgo-search
from duckduckgo_search import DDGS
def web_match_data(team_a, team_b):
    q = f"{team_a} vs {team_b} recent form tactics stats"
    hits = DDGS().text(q, max_results=5)
    return "\n".join(h["body"] for h in hits)
```
Wire it into `scout_node` (LangGraph) and the `t_scout` task (CrewAI).

**2. Add a 5th specialist — an Injury Analyst — to *both* builds.** In CrewAI that is one more `Agent` + `Task`; in LangGraph it is a new node + an edge. Notice how differently each framework makes you think.

**3. Make the LangGraph loop fire.** Set `TEAM_A, TEAM_B = "Spain", "Japan"` (no mock data), re-run Build 1, and watch `need_more` route back to `scout` — the pass counter climbs to the cap, then proceeds.

**4. Swap the provider** in Step 2 (OpenAI ⇄ Gemini ⇄ Anthropic) and compare tone, speed, and cost on the identical task.

<div style="font-family:-apple-system,Segoe UI,sans-serif;color:#5a6b82;font-size:13px;margin-top:14px;">
Slides and resources: <b>ai2lab.ai</b> · Repo: <b>github.com/mdehghani86/graph-to-crew-lab</b>
</div>